**Przetwarzanie strumieni danych lista 4 Oliwia Borkowska**

**Zad. 1** Przygotuj w Pythonie kod, który wygeneruje sygnał sinusoidalny o możliwej do zmiany częstotliwości f oraz częstotliwości próbkowania fs. Przygotuj wykres z sygnałem i próbkami pobranymi z zadaną częstotliwością próbkowania fs.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
from IPython.display import display

def sygnal_sinusoidalny(f=2.0, fs=20.0, amplituda=1.0, faza=0.0):
    t = np.arange(0, 1, 1/fs) # 1 sekunda sygnału
    y = amplituda * np.sin(2 * np.pi * f * t + faza)

    t_cont = np.linspace(0, 1, 1000) # sygnał ciągły
    y_cont = amplituda * np.sin(2 * np.pi * f * t_cont + faza) # uwzględnienie amplitudy i fazy

    nyquist_freq = fs / 2 # sprawdzanie aliasingu
    aliasing = f > nyquist_freq

    plt.figure(figsize=(10, 5))
    plt.plot(t_cont, y_cont, label='Sygnał ciągły', color='magenta')
    plt.stem(t, y, linefmt='pink', markerfmt='pink', basefmt='k-', label='Próbki')
    
    if aliasing: # aliasing
        alias_freq = abs(f % fs - fs) if f % fs > fs/2 else f % fs

        y_alias = amplituda * np.sin(2 * np.pi * alias_freq * t_cont + faza)
        plt.plot(t_cont, y_alias, 'red', label=f'Sygnał aliasowany ({alias_freq:.2f} Hz)')
        
        plt.axvspan(0, 1, alpha=0.2, color='red') # ostrzeżenie o aliasingu
        plt.text(0.5, 0.95, f" Aliasing! (f > fs/2)\nCzęstotliwość Nyquista: {nyquist_freq:.2f} Hz", 
                horizontalalignment='center', verticalalignment='top', 
                transform=plt.gca().transAxes, bbox=dict(facecolor='red', alpha=0.2))
    else:
        plt.axvspan(0, 1, alpha=0.1, color='green')
        plt.text(0.5, 0.95, f"Prawdiłowe próbkowanie (f < fs/2)\nCzęstotliwość Nyquista: {nyquist_freq:.2f} Hz", 
                horizontalalignment='center', verticalalignment='top', 
                transform=plt.gca().transAxes, bbox=dict(facecolor='green', alpha=0.2))
        
    plt.xlabel('Czas [s]', fontsize=12)
    plt.ylabel('Amplituda', fontsize=12)
    plt.title(f'Sygnał sinusoidalny (f={f:.2f} Hz, fs={fs:.2f} Hz, A={amplituda:.2f}, φ={faza:.2f} rad)', fontsize=14)
    plt.legend()
    plt.grid(True)
    plt.show()

display(interact(
    sygnal_sinusoidalny,
    f=FloatSlider(value=2.0, min=0.1, max=20.0, step=0.1, description='f [Hz]:'),
    fs=FloatSlider(value=20.0, min=1.0, max=100.0, step=1.0, description='fs [Hz]:'),
    amplituda=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='amplituda:'),
    faza=FloatSlider(value=0.0, min=0, max=2*np.pi, step=0.1, description='faza [rad]:')
))


interactive(children=(FloatSlider(value=2.0, description='f [Hz]:', max=20.0, min=0.1), FloatSlider(value=20.0…

<function __main__.sygnal_sinusoidalny(f=2.0, fs=20.0, amplituda=1.0, faza=0.0)>

**Zad 2** Przygotuj w Pythonie kod, który dokona odcinkami liniowej interpolacji (np. funkcją piecewise dostępną w pakiecie numpy) danych zebranych w zadaniu 1. Wyświetl przebieg błędu interpolacji.

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
from IPython.display import display

def sin_interpolacja_piecewise(f=2.0, fs=20.0):
    t = np.arange(0, 1, 1/fs)
    y = np.sin(2 * np.pi * f * t)

    t_cont = np.linspace(0, 1, 1000)
    y_cont = np.sin(2 * np.pi * f * t_cont)
    
    warunki = [] # lista warunków
    funkcje = []
    
    for i in range(len(t)-1):
        # warunek: t[i] <= t_cont < t[i+1]
        warunki.append((t_cont >= t[i]) & (t_cont < t[i+1]))
        
        slope = (y[i+1] - y[i]) / (t[i+1] - t[i])         # funkcja liniowa dla danego przedziału
        intercept = y[i] - slope * t[i]
        funkcje.append(lambda x, a=slope, b=intercept: a * x + b)
    
    warunki.append(t_cont >= t[-1]) # dla punktów poza ostatnią próbką
    funkcje.append(lambda x: y[-1])

    y_interp = np.piecewise(t_cont, warunki, funkcje) # funkcja numpy.piecewise
    error = y_cont - y_interp # błąd interpolacji
    
    mse = np.mean(error**2) # obliczenie błędu średniokwadratowego
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8)) # wykres sygnału oryginalnego i interpolacji (wraz z próbkami)
    ax1.plot(t_cont, y_cont, 'blue', label='Sygnał ciągły')
    ax1.plot(t_cont, y_interp, 'skyblue', label='Interpolacja liniowa (np.piecewise)')
    ax1.stem(t, y, linefmt='purple', markerfmt='purple', basefmt='k-', label='Próbki')
    ax1.set_xlabel('Czas [s]')
    ax1.set_ylabel('Amplituda')
    ax1.set_title(f'Sygnał sinusoidalny (f={f:.2f} Hz, fs={fs:.2f} Hz) i jego interpolacja')
    ax1.legend()
    ax1.grid(True)

    ax2.plot(t_cont, error, 'skyblue') # wykres błędu interpolacji
    ax2.set_xlabel('Czas [s]')
    ax2.set_ylabel('Błąd')
    ax2.set_title(f'Błąd interpolacji (MSE = {mse:.6f})')
    ax2.grid(True)
    
    # Wyświetlenie wartości MSE w dolnej części wykresu
    plt.figtext(0.5, 0.01, f'Błąd średniokwadratowy (MSE): {mse:.6f}', 
                ha='center', bbox={'facecolor':'skyblue', 'alpha':0.5, 'pad':5})
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

display(interact(
    sin_interpolacja_piecewise,
    f=FloatSlider(value=2.0, min=0.1, max=10.0, step=0.1, description='f[Hz]'),
    fs=FloatSlider(value=20.0, min=2.0, max=100.0, step=1.0, description='fs [Hz]')
))


interactive(children=(FloatSlider(value=2.0, description='f[Hz]', max=10.0, min=0.1), FloatSlider(value=20.0, …

<function __main__.sin_interpolacja_piecewise(f=2.0, fs=20.0)>

**Zad 3** Przygotuj w Pythonie kod, który dokona interpolacji punktów z zadania 1 z wykorzystaniem równania Whittakera–Shannona:
 
gdzie sinc to funkcja sinus cardinalis (funkcja sinc dostępna jest m.in. w pakiecie scipy). 
Wyświetl przebieg błędu interpolacji z wykorzystaniem równania Whittakera–Shannona


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Checkbox
from IPython.display import display


def interpolacja_whittaker_shannon(f=2.0, fs=20.0, pokaz_skladowe=False):
    T = 1/fs

    t = np.arange(0, 1, T) # 1 sek
    y = np.sin(2 * np.pi * f * t)

    t_cont = np.linspace(0, 1, 1000)
    y_cont = np.sin(2 * np.pi * f * t_cont)
    
    y_interp = np.zeros_like(t_cont) # interpolacja metodą Whittakera-Shannona
    
    skladowe = [] # ze składowymi
    
    for i in range(len(t)): # ograniczenie sumy do faktycznych próbek
        sincArg = (t_cont - t[i]) / T
        skladowa = y[i] * np.sinc(sincArg)
        y_interp += skladowa

        if pokaz_skladowe:
            skladowe.append(skladowa)

    error = y_cont - y_interp # bład interpolacji


    if pokaz_skladowe: # liczba wykresów zależności czy pokazać składowe, czy nie
        fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 12))
    else:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
    

    ax1.plot(t_cont, y_cont, 'orange', label='Sygnał ciągły') # wykres sygnału i interpolacji oraz błędu
    ax1.plot(t_cont, y_interp, 'yellow', label='Interpolacja Whittakera-Shannona')
    ax1.stem(t, y, linefmt='red', markerfmt='red', basefmt='k-', label='Próbki')
    ax1.set_xlabel('Czas [s]')
    ax1.set_ylabel('Amplituda')
    ax1.set_title(f'Sygnał sinusoidalny (f={f:.2f} Hz, fs={fs:.2f} Hz) i jego interpolacja')
    ax1.legend()
    ax1.grid(True)
    
    ax2.plot(t_cont, error, 'r-')
    ax2.set_xlabel('Czas [s]')
    ax2.set_ylabel('Błąd')
    ax2.set_title('Błąd interpolacji') # rzonica między sygnałem ciągłym a interpolowanym
    ax2.grid(True)
    
    if pokaz_skladowe: # jeśli jest wbrane to pokaż składowe
        for i, skladowa in enumerate(skladowe):

            ax3.plot(t_cont, skladowa, 'k--', alpha=0.3) # składowe
        
        ax3.stem(t, y, linefmt='r-', markerfmt='ro', basefmt='k-', label='Próbki')
        
        ax3.plot(t_cont, y_interp, 'g-', linewidth=2, label='Suma składowych') # suma składowych
        
        ax3.set_xlabel('Czas [s]')
        ax3.set_ylabel('Amplituda')
        ax3.set_title('Poszczególne składowe interpolacji (funkcje sinc)')
        ax3.legend()
        ax3.grid(True)
    
    plt.tight_layout()
    plt.show()
    
display(interact(
    interpolacja_whittaker_shannon,
    f=FloatSlider(value=2.0, min=0.1, max=10.0, step=0.1, description='f [Hz]'),
    fs=FloatSlider(value=20.0, min=2.0, max=100.0, step=1.0, description='fs [Hz]'),
    pokaz_skladowe=Checkbox(value=False, description='Pokaż składowe')
))


interactive(children=(FloatSlider(value=2.0, description='f [Hz]', max=10.0, min=0.1), FloatSlider(value=20.0,…

<function __main__.interpolacja_whittaker_shannon(f=2.0, fs=20.0, pokaz_skladowe=False)>

In [19]:
# 127a D2 - sala

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
from scipy import signal  # nie stosujemy
from IPython.display import display


def interpolacja_whittaker_shannon(f=2.0, fs=20.0):
    T = 1/fs

    t = np.arange(0, 1, T) # 1 sek
    y = np.sin(2 * np.pi * f * t)

    t_cont = np.linspace(0, 1, 1000)
    y_cont = np.sin(2 * np.pi * f * t_cont)
    
    y_interp = np.zeros_like(t_cont) # interpolacja metodą Whittakera-Shannona
    
    for i in range(len(t)): # ograniczenie sumy do faktycznych próbek
        sincArg = (t_cont - t[i]) / T
        y_interp += y[i] * np.sinc(sincArg)

    error = y_cont - y_interp # bład interpolacji


    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8)) # wykresy
    
    ax1.plot(t_cont, y_cont, 'b-', label='Sygnał ciągły')
    ax1.plot(t_cont, y_interp, 'g-', label='Interpolacja Whittakera-Shannona')
    ax1.stem(t, y, linefmt='r-', markerfmt='ro', basefmt='k-', label='Próbki')
    ax1.set_xlabel('Czas [s]')
    ax1.set_ylabel('Amplituda')
    ax1.set_title(f'Sygnał sinusoidalny (f={f:.2f} Hz, fs={fs:.2f} Hz) i jego interpolacja')
    ax1.legend()
    ax1.grid(True)
    
    ax2.plot(t_cont, error, 'r-')
    ax2.set_xlabel('Czas [s]')
    ax2.set_ylabel('Błąd')
    ax2.set_title('Błąd interpolacji (różnica między sygnałem ciągłym a interpolowanym)')
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()
    
display(interact(
    interpolacja_whittaker_shannon,
    f=FloatSlider(value=2.0, min=0.1, max=10.0, step=0.1, description='f [Hz]'),
    fs=FloatSlider(value=20.0, min=2.0, max=100.0, step=1.0, description='fs [Hz]')
))


interactive(children=(FloatSlider(value=2.0, description='f [Hz]', max=10.0, min=0.1), FloatSlider(value=20.0,…

<function __main__.interpolacja_whittaker_shannon(f=2.0, fs=20.0)>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def sygnal_sinusoidalny(f=2.0, fs=20.0):
    t = np.arange(0, 1, 1/fs)  # czas próbkowania - 1 sekunda sygnału
    y = np.sin(2 * np.pi * f * t)

    t_cont = np.linspace(0, 1, 1000)  # sygnał ciągły
    y_cont = np.sin(2 * np.pi * f * t_cont)

    plt.figure(figsize=(10, 5))
    plt.plot(t_cont, y_cont, label='Sygnał ciągły', color='magenta')
    plt.stem(t, y, linefmt='pink', markerfmt='pink', basefmt='k-', label='Próbki')

    plt.xlabel('Czas [s]', fontsize=12)
    plt.ylabel('Amplituda', fontsize=12)
    plt.title(f'Sygnał sinusoidalny (f={f:.2f} Hz, fs={fs:.2f} Hz)', fontsize=14)
    plt.legend()
    plt.grid(True)
    plt.show()

# suwaki
interact(
    sygnal_sinusoidalny,
    f=FloatSlider(value=2.0, min=0.1, max=10.0, step=0.1, description='f [Hz]'),
    fs=FloatSlider(value=20.0, min=2.0, max=100.0, step=1.0, description='fs [Hz]')
)
